# ATML PA3 - Task 1: Large Language Model Fine-Tuning on Kaggle

This notebook runs the complete experimental pipeline for **Task 1: LLM Fine-Tuning** (`Qwen/Qwen3-0.6B`) on **Kaggle GPUs** (NVIDIA Tesla T4 or P100 16GB).

### Kaggle Settings Checklist:
- **Accelerator**: `GPU T4 x 2` or `GPU P100` (16GB VRAM)
- **Internet**: `On` (required for downloading dataset & model checkpoints from Hugging Face)
- **Persistence**: `Files only` or `Variables and Files`

## 1. Verify GPU Hardware and PyTorch CUDA Environment

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 2. Clone Repository & Setup Workspace

In [ ]:
import os

# Clean up any stray redirection files if previously created
!rm -f =*

# Clone the kaggle-gpu branch into ATML-PA3 if not already present
if not os.path.exists("/kaggle/working/ATML-PA3") and not os.path.exists("task1"):
    !git clone -b kaggle-gpu https://github.com/supus4562/ATML-PA3.git /kaggle/working/ATML-PA3

if os.path.exists("/kaggle/working/ATML-PA3"):
    %cd /kaggle/working/ATML-PA3

print("Current working directory:", os.getcwd())
!ls -la

## 3. Install Project Dependencies
Install the required packages with quotes to avoid shell redirection operators.

In [ ]:
!pip install -q "transformers>=4.45.0" "datasets>=2.20.0" "accelerate>=0.34.0" "peft>=0.12.0" pyyaml pandas matplotlib

## 4. Run the Full End-to-End Pipeline
Execute all 10 stages using the Kaggle-optimized profile:
1. Data filtering & tokenization cache (40,000 source order, GSM8K leak filtered)
2. Baseline evaluation (Update 0)
3. Full Fine-Tuning (midpoint 625 & final 1250)
4. Full FT + L2-SP regularized fine-tuning (final 1250)
5. LoRA-4 (final 1250)
6. LoRA-16 (final 1250)
7. OPLoRA-16 with rank-128 SVD subspace protection (final 1250)
8. Weight-space ($\rho_k, \eta$) and activation ratio ($R(z)$) diagnostics
9. Qualitative prompt suite (24 prompts, 168 responses, 42-response audit)
10. Synthesis & metrics summary generation

In [ ]:
# Execute with the Kaggle launcher script
!bash run_kaggle.sh

## 5. (Alternative) Run Individual Stages
You can also run specific stages on Kaggle independently using `--stages`:

In [ ]:
# Example: Run only baseline and LoRA-16
# !python3 -m task1.run_all --config configs/kaggle_config.yaml --stages baseline train_lora16

# Example: Run only diagnostics, qualitative, and synthesis after checkpoints exist
# !python3 -m task1.run_all --config configs/kaggle_config.yaml --stages diagnostics qualitative synthesize

## 6. Inspect Results and Benchmark Deltas

In [ ]:
import json
import pandas as pd

summary_file = "results/task1/metrics_summary.json"
if os.path.exists(summary_file):
    with open(summary_file, "r") as f:
        data = json.load(f)
    
    deltas_df = pd.DataFrame(data.get("benchmark_deltas_pp", []))
    print("\n--- Consolidated Benchmark Scores & Baseline Deltas (pp) ---")
    display(deltas_df)
else:
    print("Results summary not yet found. Run the pipeline above to generate.")

## 7. Visualize Learning Curves

In [ ]:
import matplotlib.pyplot as plt

curves_file = "results/task1/learning_curves.csv"
if os.path.exists(curves_file):
    df_curves = pd.read_csv(curves_file)
    
    plt.figure(figsize=(10, 5))
    for variant, group in df_curves.groupby("variant"):
        plt.plot(group["update"], group["train_loss"], label=variant, alpha=0.8)
    
    plt.title("Training Loss vs Update Steps (Effective Batch Size = 32)")
    plt.xlabel("Update Step")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.show()
else:
    print("Learning curves file not yet found.")